# Tomahawk G2 — Run 2 checkpoint diagnostic

Attach **Tomahawk_G2_Run_2_v2_checkpoint.zip** as your only Kaggle Input, enable Internet if DESC is not installed, and select **Run All**. This notebook reads the saved L12 and L16 equilibria without rerunning a solve or requiring the large MGRID ZIP. It compares the saved plasma boundary to CAD by matching theta and by matching physical R/Z points at each toroidal angle, then writes **Tomahawk_G2_Run_2_Diagnostic.zip** to `/kaggle/working`.

The previous Run 3 notebook requires a complete Run 2 checkpoint. The uploaded Run 2 checkpoint is incomplete because L16 ended with process exit `-9`. The L12 solve also missed its numerical gates. This diagnostic determines whether the reported multi-meter CAD gap is geometric or a theta coordinate comparison artifact; it does not declare G2 closure.


In [1]:
"""Fast, checkpoint-only Kaggle diagnostic for the incomplete G2 Run 2."""
from __future__ import annotations

import hashlib
import importlib.util
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

INPUT_NAME = "Tomahawk_G2_Run_2_v2_checkpoint.zip"
SOURCE_SHA = "86b9481410b1b0cacf00a3b20733827722742d4843b371987de92cbb0191bb8e"
ENGINE_SHA = "6bf8ea82626f6c3d485fef673ea12de6d4a5074189188bb059f8c71371fae043"
input_root = Path(os.environ.get("TOMAHAWK_INPUT_ROOT", "/kaggle/input"))
work_root = Path(os.environ.get("TOMAHAWK_WORK_ROOT", "/kaggle/working"))
work_root.mkdir(parents=True, exist_ok=True)


def digest_bytes(data):
    return hashlib.sha256(data).hexdigest()


archives = sorted(input_root.rglob(INPUT_NAME)) if input_root.is_dir() else []
if len(archives) != 1:
    raise FileNotFoundError(
        "Attach exactly one " + INPUT_NAME + " as a Kaggle Input dataset. "
        "This diagnostic does not need the 425 MB source ZIP."
    )

required = {
    "run_status.json",
    "steps/L12/metrics.json",
    "steps/L12/fixed_equilibrium.h5",
    "steps/L12/free_equilibrium.h5",
    "steps/L16/fixed_equilibrium.h5",
    "L16.log",
}
with zipfile.ZipFile(archives[0]) as archive:
    if len(archive.namelist()) != len(set(archive.namelist())):
        raise ValueError("Duplicate archive members")
    if not required.issubset(archive.namelist()):
        raise ValueError("Missing Run 2 status, saved L12/L16 states, or logs")
    status = json.loads(archive.read("run_status.json"))
    if (
        status.get("stage"),
        status.get("complete"),
        status.get("source_manifest_sha256"),
        status.get("engine_sha256"),
    ) != (2, False, SOURCE_SHA, ENGINE_SHA):
        raise ValueError("Checkpoint is not the failed G2 Run 2 v2 being diagnosed")
    metrics = json.loads(archive.read("steps/L12/metrics.json"))
    log = archive.read("L16.log").decode("utf-8", errors="replace")
    if metrics.get("resolution") != [12, 12, 3]:
        raise ValueError("Unexpected L12 state")
    state_files = {}
    folder = work_root / "g2_run2_diagnostic_states"
    folder.mkdir(exist_ok=True)
    for label, member in {
        "L12_fixed": "steps/L12/fixed_equilibrium.h5",
        "L12_free": "steps/L12/free_equilibrium.h5",
        "L16_fixed": "steps/L16/fixed_equilibrium.h5",
    }.items():
        data = archive.read(member)
        if not data.startswith(b"\x89HDF\r\n\x1a\n"):
            raise ValueError(member + " is not HDF5")
        path = folder / (label + ".h5")
        path.write_bytes(data)
        state_files[label] = (path, digest_bytes(data))

steps = status["steps"]
print("RUN 2 STATUS:", json.dumps(steps, indent=2))
print("L12 saved metrics:", json.dumps(metrics, indent=2))
print("L16 log tail:", log[-1200:])
print("Saved states:", {k: v[1] for k, v in state_files.items()})

if os.environ.get("TOMAHAWK_ARCHIVE_ONLY") == "1":
    print("Archive-only local validation complete; DESC geometry check skipped.")
else:
    os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
    if importlib.util.find_spec("desc") is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "desc-opt==0.17.3"]
        )
    import desc
    import numpy as np
    from scipy.spatial import cKDTree
    from desc.equilibrium import Equilibrium
    from desc.grid import LinearGrid

    if not desc.__version__.startswith("0.17.3"):
        raise RuntimeError("DESC 0.17.3 required; found " + desc.__version__)

    theta = np.linspace(0.0, 2 * np.pi, 128, endpoint=False)
    zeta = np.linspace(0.0, 2 * np.pi / 16, 16, endpoint=False)

    def compare_boundary(eq):
        if int(eq.NFP) != 16:
            raise ValueError("Equilibrium has the wrong field period")
        grid = LinearGrid(rho=1.0, theta=theta, zeta=zeta, NFP=16)
        data = eq.compute(["R", "Z"], grid=grid)
        nodes = np.asarray(grid.nodes)
        model = np.column_stack(
            (np.asarray(data["R"]).reshape(-1), np.asarray(data["Z"]).reshape(-1))
        )
        if model.shape != (len(nodes), 2) or not np.isfinite(model).all():
            raise ValueError("DESC returned invalid boundary coordinates")
        t, p = nodes[:, 1], nodes[:, 2]
        reference = np.column_stack(
            (
                7.5 + 3.0 * np.cos(t) + 0.01 * np.cos(t - 16 * p),
                2.138969432449380 * np.sin(t)
                + 0.01 * np.sin(t - 16 * p),
            )
        )
        paired_dist = np.linalg.norm(model - reference, axis=1)
        nearest = []
        for phi in zeta:
            indices = np.flatnonzero(np.isclose(p, phi, atol=1e-10, rtol=0))
            if len(indices) != len(theta):
                raise ValueError("Unexpected DESC boundary grid ordering")
            a = model[indices]
            b = reference[indices]
            nearest.extend(cKDTree(b).query(a)[0])
            nearest.extend(cKDTree(a).query(b)[0])
        nearest = np.asarray(nearest)
        return {
            "same_theta_rms_m": float(np.sqrt(np.mean(paired_dist**2))),
            "same_theta_max_m": float(paired_dist.max()),
            "shape_nearest_symmetric_rms_m": float(np.sqrt(np.mean(nearest**2))),
            "shape_nearest_symmetric_max_m": float(nearest.max()),
            "R_min_m": float(model[:, 0].min()),
            "R_max_m": float(model[:, 0].max()),
            "Z_min_m": float(model[:, 1].min()),
            "Z_max_m": float(model[:, 1].max()),
        }

    shapes = {}
    for label, (path, _) in state_files.items():
        eq = Equilibrium.load(str(path))
        shapes[label] = compare_boundary(eq)
        print(label, json.dumps(shapes[label], indent=2), flush=True)
        del eq

    result = {
        "checkpoint_sha256": hashlib.sha256(archives[0].read_bytes()).hexdigest(),
        "checkpoint_status": status,
        "L12_metrics": metrics,
        "boundary_comparison": shapes,
        "saved_state_sha256": {k: v[1] for k, v in state_files.items()},
        "L16_log_tail": log[-3000:],
        "interpretation_note": (
            "Compare same-theta and geometric nearest distances. Neither a "
            "small geometric distance nor optimizer termination alone closes G2. "
            "Exit -9 is a SIGKILL; OOM is a likely but unconfirmed cause."
        ),
    }
    report = work_root / "Tomahawk_G2_Run_2_Diagnostic.json"
    report.write_text(json.dumps(result, indent=2, allow_nan=False))
    output_zip = work_root / "Tomahawk_G2_Run_2_Diagnostic.zip"
    with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as out:
        out.write(report, report.name)
        out.writestr("L16.log", log)
    print("DOWNLOAD:", output_zip, flush=True)


FileNotFoundError: Attach exactly one Tomahawk_G2_Run_2_v2_checkpoint.zip as a Kaggle Input dataset. This diagnostic does not need the 425 MB source ZIP.